# Quoridor AI — POC Training (5×5)

**Group 501** | Colman College | DL Final Project

AlphaZero-inspired agent for Quoridor trained via self-play.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run cells in order
3. If Colab disconnects, re-run from **Section 1** — `resume=True` picks up automatically

---
## 1. Environment Setup
Run once per Colab session.

In [1]:
# 1.1 — Verify GPU
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be slow.")
    print("Go to Runtime → Change runtime type → T4 GPU")

GPU: Tesla T4
Memory: 15.6 GB


In [2]:
# ── Anti-disconnect for Colab ──
# This JavaScript pings the Colab runtime to prevent idle timeout
import IPython
IPython.display.display(IPython.display.Javascript('''
function KeepAlive() {
    console.log("Keeping alive " + new Date().toLocaleTimeString());
    document.querySelector("colab-toolbar-button#connect")?.click();
}
setInterval(KeepAlive, 60000);
'''))
print("Anti-disconnect activated (pings every 60s)")


<IPython.core.display.Javascript object>

Anti-disconnect activated (pings every 60s)


In [ ]:
# 1.2 — Mount Google Drive (checkpoints & logs persist here)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

DRIVE_ROOT = "/content/drive/MyDrive/DL-Quoridor"
CHECKPOINT_DIR = f"{DRIVE_ROOT}/checkpoints"
LOG_DIR = f"{DRIVE_ROOT}/logs"
NOTEBOOK_DIR = f"{DRIVE_ROOT}/notebooks"

!mkdir -p "{CHECKPOINT_DIR}" "{LOG_DIR}" "{NOTEBOOK_DIR}"
print(f"Checkpoints: {CHECKPOINT_DIR}")
print(f"Logs:        {LOG_DIR}")

Mounted at /content/drive
Checkpoints: /content/drive/MyDrive/DL-Quoridor/checkpoints
Logs:        /content/drive/MyDrive/DL-Quoridor/logs


In [4]:
# 1.3 — Clone repo and install dependencies
import os
import importlib

REPO_DIR = "/content/dl-quoridor"
BRANCH = "training/poc-5x5"  # change if needed

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git fetch --all
    !git checkout {BRANCH}
    !git pull
    print("\nRepo updated.")
else:
    !git clone https://github.com/ReefKenig/dl-quoridor.git {REPO_DIR}
    %cd {REPO_DIR}
    !git checkout {BRANCH}
    print("\nRepo cloned.")

!pip install -r requirements.txt -q

# Force reload all src modules so code changes from git pull take effect
import src.utils.checkpoint, src.utils.logger, src.utils.config
import src.env.quoridor_env, src.env.tensor_spec
import src.mcts.mcts, src.mcts.evaluator, src.mcts.self_play
import src.model.network
for mod in [
    src.utils.checkpoint, src.utils.logger, src.utils.config,
    src.env.quoridor_env, src.env.tensor_spec,
    src.mcts.mcts, src.mcts.evaluator, src.mcts.self_play,
    src.model.network,
]:
    importlib.reload(mod)

print("Dependencies installed & modules reloaded.")

Cloning into '/content/dl-quoridor'...
remote: Enumerating objects: 409, done.
remote: Counting objects: 100% (409/409), done.
remote: Compressing objects: 100% (233/233), done.
remote: Total 409 (delta 236), reused 332 (delta 167), pack-reused 0 (from 0)
Receiving objects: 100% (409/409), 113.10 KiB | 1.22 MiB/s, done.
Resolving deltas: 100% (236/236), done.
/content/dl-quoridor
Branch 'training/poc-5x5' set up to track remote branch 'training/poc-5x5' from 'origin'.
Switched to a new branch 'training/poc-5x5'

Repo cloned.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.3/101.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 9.9 MB/s eta 0:00:00
Dependencies installed & modules reloaded.


---
## 2. Validation
Confirm all components work before training.

In [ ]:
# 2.1 — Smoke test: full pipeline (env → tensor → model → MCTS → self-play)
from src.env.quoridor_env import QuoridorEnv
from src.model.network import QuoridorModel
from src.mcts.mcts import MCTS, MCTSConfig
from src.mcts.self_play import play_one_game

env = QuoridorEnv(is_poc=True)
model = QuoridorModel(board_size=5, action_space_size=44, num_channels=16, num_res_blocks=2)

def nn_evaluate(state):
    tensor = env.state_to_tensor(state)
    return model.predict(tensor)

mcts = MCTS(config=MCTSConfig(num_simulations=50), evaluate_fn=nn_evaluate)
samples, winner = play_one_game(env, mcts, max_moves=100)

print(f"Game: {len(samples)} moves, winner=P{winner}")
print(f"Tensor shape: {samples[0].state.shape}")
print(f"Policy shape: {samples[0].policy_target.shape}")
assert samples[0].state.shape == (5, 5, 10)
assert samples[0].policy_target.shape == (44,)
print("\n✓ Smoke test passed — pipeline is connected.")

Game: 65 moves, winner=P0
Tensor shape: (5, 5, 10)
Policy shape: (44,)

✓ Smoke test passed — pipeline is connected.


In [ ]:
# 2.2 — MCTS (random rollouts) vs random agent on real env
from src.mcts.evaluator import evaluate_against_random, mcts_agent

env = QuoridorEnv(is_poc=True)
mcts_random = MCTS(config=MCTSConfig(num_simulations=200))
agent = mcts_agent(mcts_random, temperature=0.1)

result = evaluate_against_random(env, agent, num_games=20)
print(result.summary())
print(f"\n✓ MCTS wins {result.agent_a_win_rate:.0%} vs random (expected >60%)")

KeyboardInterrupt: 

---
## 3. Training

In [ ]:
# 3 — Full POC training (12 iterations, ~15 hours)
# Re-run this cell after Colab disconnects — resume=True picks up from last checkpoint

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(levelname)s: %(message)s",
    force=True,
)

from src.env.quoridor_env import QuoridorEnv
from src.model.network import QuoridorModel
from src.mcts.self_play import training_loop, TrainingConfig

env = QuoridorEnv(is_poc=True)
model = QuoridorModel(
    board_size=5, action_space_size=44,
    num_channels=64, num_res_blocks=4,
)

config = TrainingConfig(
    num_iterations=12,
    games_per_iteration=100,
    mcts_simulations=400,
    batch_size=256,
    training_epochs=50,
    eval_games=40,
    win_threshold=0.55,
    replay_buffer_size=50000,
    self_play_checkpoint_freq=10,
    max_game_moves=80,
)

training_loop(
    env=env,
    model=model,
    config=config,
    checkpoint_dir=CHECKPOINT_DIR,
    log_dir=LOG_DIR,
    use_wandb=False,
    resume=True,
)

2026-03-17 08:45:44,189 [src.model.network] INFO: QuoridorModel: 305995 params, device=cuda, board=5x5, actions=44
2026-03-17 08:45:46,543 [src.utils.checkpoint] INFO: Loaded checkpoint: iter=2, buffer=11369
2026-03-17 08:45:47,257 [src.model.network] INFO: Model loaded from /content/drive/MyDrive/DL-Quoridor/checkpoints/iter_0002/model.pt
2026-03-17 08:45:47,258 [src.mcts.self_play] WARNING: MID-ITERATION RESUME: iteration 2, resuming from game 90/100 (skipping 90 already-played games)
2026-03-17 08:45:47,260 [src.mcts.self_play] INFO: Checkpoint loaded: iteration=2, buffer=11369 samples, best_wr=0.0%
2026-03-17 08:45:47,260 [src.mcts.self_play] INFO: ==================================================
2026-03-17 08:45:47,262 [src.mcts.self_play] INFO: Iteration 3/12
2026-03-17 08:45:47,263 [src.mcts.self_play] INFO:   Resuming self-play from game 90/100 (10 remaining)...
2026-03-17 08:52:48,681 [src.mcts.self_play] INFO:     Games: 100/100
2026-03-17 08:52:48,682 [src.mcts.self_play] 

KeyboardInterrupt: 

---
## 4. Monitor & Analyze Results

In [ ]:
# 4.1 — Training curves
import json
import matplotlib.pyplot as plt

log_path = f"{LOG_DIR}/metrics_full.json"

try:
    with open(log_path) as f:
        metrics = json.load(f)
except FileNotFoundError:
    print("No metrics yet — run training first.")
    metrics = []

if metrics:
    iters = [m['iteration'] for m in metrics]
    loss_p = [m['loss_policy'] for m in metrics]
    loss_v = [m['loss_value'] for m in metrics]
    wr = [m['win_rate_vs_random'] for m in metrics]
    avg_len = [m['avg_game_length'] for m in metrics]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Quoridor 5×5 POC — Training Progress', fontsize=14)

    axes[0, 0].plot(iters, loss_p, 'b-o', markersize=3)
    axes[0, 0].set_title('Policy Loss')
    axes[0, 0].set_xlabel('Iteration')
    axes[0, 0].set_ylabel('Cross-Entropy')
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(iters, loss_v, 'r-o', markersize=3)
    axes[0, 1].set_title('Value Loss')
    axes[0, 1].set_xlabel('Iteration')
    axes[0, 1].set_ylabel('MSE')
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(iters, [w * 100 for w in wr], 'g-o', markersize=3)
    axes[1, 0].axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='Random baseline')
    axes[1, 0].set_title('Win Rate vs Random Agent')
    axes[1, 0].set_xlabel('Iteration')
    axes[1, 0].set_ylabel('Win Rate (%)')
    axes[1, 0].set_ylim(0, 105)
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(iters, avg_len, 'm-o', markersize=3)
    axes[1, 1].set_title('Average Game Length')
    axes[1, 1].set_xlabel('Iteration')
    axes[1, 1].set_ylabel('Moves')
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{LOG_DIR}/training_curves.png", dpi=150)
    plt.show()

    print(f"\nLatest iteration: {iters[-1]}")
    print(f"Policy loss: {loss_p[-1]:.4f}")
    print(f"Value loss:  {loss_v[-1]:.4f}")
    print(f"Win rate:    {wr[-1]:.1%}")

In [ ]:
# 4.2 — Metrics table
if metrics:
    print(f"{'Iter':>4} | {'Loss_P':>8} | {'Loss_V':>8} | {'WR_Random':>10} | {'Avg_Len':>8} | {'Best':>5}")
    print("-" * 60)
    for m in metrics:
        print(
            f"{m['iteration']:4d} | "
            f"{m['loss_policy']:8.4f} | "
            f"{m['loss_value']:8.4f} | "
            f"{m['win_rate_vs_random']:9.1%} | "
            f"{m['avg_game_length']:8.1f} | "
            f"{'*' if m.get('model_accepted') else ''}"
        )

---
## 5. Evaluate Best Model

In [ ]:
# 5.1 — Load best model and evaluate thoroughly
from src.env.quoridor_env import QuoridorEnv
from src.model.network import QuoridorModel
from src.mcts.mcts import MCTS, MCTSConfig
from src.mcts.evaluator import evaluate_against_random, mcts_agent
import os

env = QuoridorEnv(is_poc=True)
best_model = QuoridorModel(board_size=5, action_space_size=44)

best_path = f"{CHECKPOINT_DIR}/best/model.pt"
if os.path.exists(best_path):
    best_model.load(best_path)
    print(f"Loaded best model from {best_path}")
else:
    print("No best model found — run training first.")

def best_nn_evaluate(state):
    tensor = env.state_to_tensor(state)
    return best_model.predict(tensor)

# Evaluate at different MCTS simulation counts
for sims in [100, 200, 400]:
    mcts = MCTS(
        config=MCTSConfig(num_simulations=sims),
        evaluate_fn=best_nn_evaluate,
    )
    agent = mcts_agent(mcts, temperature=0.1)
    result = evaluate_against_random(env, agent, num_games=50)
    print(f"  {sims:>4} sims: {result.summary()}")

In [ ]:
# 5.2 — Compare: trained model vs untrained model
from src.mcts.evaluator import evaluate

untrained_model = QuoridorModel(board_size=5, action_space_size=44)

def untrained_evaluate(state):
    tensor = env.state_to_tensor(state)
    return untrained_model.predict(tensor)

mcts_trained = MCTS(
    config=MCTSConfig(num_simulations=200),
    evaluate_fn=best_nn_evaluate,
)
mcts_untrained = MCTS(
    config=MCTSConfig(num_simulations=200),
    evaluate_fn=untrained_evaluate,
)

trained_agent = mcts_agent(mcts_trained, temperature=0.1)
untrained_agent = mcts_agent(mcts_untrained, temperature=0.1)

result = evaluate(env, agent_a=trained_agent, agent_b=untrained_agent, num_games=40)
print(f"Trained vs Untrained: {result.summary()}")